# What a miss costs, and what judged can save you

Standalone: nothing in judged depends on this notebook, and it only uses the Python standard library.
Run the cells top to bottom. Part 1 shows published prices in your currency (exchange rates are downloaded from the
European Central Bank), part 2 checks that each source page still says what the figure was taken from, part 3 asks
for **your** numbers and estimates, per option, how much money and paperwork time judged can save.

Every estimate is arithmetic on figures you typed. Nothing here is measured, and the defaults are guesses to be replaced.

In [ ]:
import json, re, urllib.request
from pathlib import Path

OFFLINE = False          # True: no downloads at all (rates and source pages are skipped)
CURRENCY = "EUR"         # the currency every figure is shown in

DATA = json.loads(Path("costdata.json").read_text())

def download(url, limit=3_000_000, tries=2):
    req = urllib.request.Request(url, headers={"User-Agent": "judged-notebook/0 public list download"})
    for n in range(tries):
        try:
            with urllib.request.urlopen(req, timeout=30) as r:
                return r.read(limit).decode("utf-8", "replace")
        except TimeoutError:
            if n == tries - 1:
                raise

def ecb_rates():
    # {currency: units per 1 EUR}; None when offline or the download fails
    if OFFLINE:
        return None
    try:
        xml = download("https://www.ecb.europa.eu/stats/eurofxref/eurofxref-daily.xml")
        rates = {"EUR": 1.0}
        for cur, rate in re.findall(r"currency='([A-Z]{3})'\s+rate='([0-9.]+)'", xml):
            rates[cur] = float(rate)
        return rates if len(rates) > 1 else None
    except (OSError, ValueError) as e:
        print("no exchange rates:", e)
        return None

RATES = ecb_rates()
print("rates:", "downloaded, %d currencies" % len(RATES) if RATES else "none, figures stay in their own currency")

## 1. Published figures in your currency

In [ ]:
def convert(amount, src, dst, rates=None):
    rates = RATES if rates is None else rates
    if src == dst:
        return amount
    if not rates or src not in rates or dst not in rates:
        return None
    return amount / rates[src] * rates[dst]

def show(f, cur=None):
    cur = cur or CURRENCY
    lo, hi = convert(f["low"], f["currency"], cur), convert(f["high"], f["currency"], cur)
    if lo is None:
        lo, hi, cur = f["low"], f["high"], f["currency"]
    txt = f"{cur} {lo:,.0f}" + (f" to {hi:,.0f}" if round(lo) != round(hi) else "")
    per = {"year": 1, "device-year": 1}.get(f["period"])
    if per:
        txt += f"   ({lo/12:,.0f} a month, {lo/365:,.2f} a day)"
    if f["period"] == "minute":
        txt += f"   ({lo*60:,.0f} to {hi*60:,.0f} an hour)"
    return txt

for f in DATA["figures"]:
    print(f"{f['label']:<70} {show(f)}  per {f['period']}  [{f['kind']}: {f['source']}]")
print("\nkind: published = survey or law; vendor = said by someone who sells the fix; computed = arithmetic on a published figure")

## 2. Do the sources still say it? (downloads each page once)

In [ ]:
def verify(figures=DATA["figures"]):
    pages, out = {}, []
    for f in figures:
        if f["url"] not in pages:
            try:
                pages[f["url"]] = download(f["url"])
            except (OSError, ValueError) as e:
                pages[f["url"]] = e
        page = pages[f["url"]]
        if isinstance(page, Exception):
            out.append((f["id"], "unreachable", str(page)[:70]))
        elif not f.get("quote"):
            out.append((f["id"], "page only", "no exact quote (%s figure): open the page and read it yourself" % f["kind"]))
        elif re.sub(r"\s+", " ", f["quote"]) in re.sub(r"\s+", " ", page):
            out.append((f["id"], "found", f["quote"]))
        else:
            out.append((f["id"], "NOT FOUND", f["quote"] + " is gone from the page: the figure may have changed"))
    return out

if not OFFLINE:
    for row in verify():
        print("%-12s %-24s %s" % (row[1], row[0], row[2]))

## 3. Your numbers: what each judged option can save

Type your own values. The two kinds of saving:

* **money** that a missed deadline or hole would cost (outage, renewed contract, breach);
* **paperwork time** judged does for you (collecting evidence for Cyber Essentials, ISO 27001, insurer questionnaires, GDPR checklists).

The chance values are *your* estimates of how often you currently miss something. Lower them to be careful.

In [ ]:
MINE = dict(
    currency="EUR",
    # the business
    revenue_per_hour=300.0,          # what you earn in an ordinary trading hour
    staff_idle_when_down=5,          # people who cannot work when the site, mail or login is down
    staff_cost_per_hour=40.0,        # loaded hourly cost of one person
    # expiry (certificates, keys, domains, licences)
    expiry_items=25,                 # how many dated things you track by hand today
    expiry_misses_per_year=0.1,      # your guess: 0.1 = one missed in ten years
    hours_to_fix_a_miss=4.0,
    # contracts with a notice period
    contracts_value_per_year=12000.0,
    contracts_missed_share=0.05,     # share of that value that renews by accident
    # Windows 10 machines that still need updates
    windows10_devices=0,
    # a breach
    breach_cost=None,                # None = the IBM global average, converted; or type your own estimate
    breach_chance_per_year=0.03,     # your guess
    breach_share_prevented=0.10,     # how much of that risk judged's findings remove: a guess, keep it low
    # paperwork
    evidence_hours_per_year=40.0,    # hours spent collecting proof for audits, insurers, customers
    evidence_share_automated=0.5,    # how much of that judged's report already holds
    office_hour_cost=45.0,
)

In [ ]:
def to_cur(x, src="EUR"):
    v = convert(x, src, MINE["currency"])
    return x if v is None else v

def fig(i):
    f = next(f for f in DATA["figures"] if f["id"] == i)
    return f

def estimate(m=MINE):
    cur = m["currency"]
    down_hour = m["revenue_per_hour"] + m["staff_idle_when_down"] * m["staff_cost_per_hour"]
    breach = m["breach_cost"] if m["breach_cost"] is not None else convert(fig("breach-global")["low"], "USD", cur)
    esu1 = convert(fig("win10-esu-1")["low"], "USD", cur)
    if breach is None or esu1 is None:
        print(f"WARNING: no USD to {cur} rate (offline?): the USD figures below are used as if they were {cur}, set breach_cost yourself")
        breach = breach if breach is not None else fig("breach-global")["low"]
        esu1 = esu1 if esu1 is not None else fig("win10-esu-1")["low"]
    rows = [
        ("Expiry tracking", "misses/year x hours x (revenue/h + idle staff x cost/h)",
         m["expiry_misses_per_year"] * m["hours_to_fix_a_miss"] * down_hour),
        ("Contract notice dates", "contract value x share that renews by accident",
         m["contracts_value_per_year"] * m["contracts_missed_share"]),
        ("Windows 10 end of life", "devices x extended-updates year 1 (cost avoided by planning the move)",
         m["windows10_devices"] * esu1),
        ("Known exploited vulnerabilities, secrets, hardening", "breach cost x chance/year x share prevented",
         breach * m["breach_chance_per_year"] * m["breach_share_prevented"]),
        ("Paperwork: evidence for audits and insurers", "hours x share automated x hour cost",
         m["evidence_hours_per_year"] * m["evidence_share_automated"] * m["office_hour_cost"]),
    ]
    return rows

rows = estimate()
total = sum(r[2] for r in rows)
print(f"{'option':<55} {'a year':>10} {'a month':>10} {'a day':>8}   formula")
for name, how, v in rows:
    print(f"{name:<55} {v:>10,.0f} {v/12:>10,.0f} {v/365:>8,.2f}   {how}")
print(f"{'TOTAL (' + MINE['currency'] + ')':<55} {total:>10,.0f} {total/12:>10,.0f} {total/365:>8,.2f}")
print("\njudged itself costs 0 (MIT licence) plus the afternoon it takes to write the config file.")

## What this does not know

* How often you really miss things: replace the guesses with your own history (a missed renewal, an expired certificate).
* The size of an outage or a breach for *your* firm: the published averages mix banks and shops of ten people.
* Fines and insurance: they depend on the country, the data and the contract. See `docs/costs.md` for the sources and the ceilings.